**Classical Boolean gates on a quantum computer.**
Quantum computers are universal: any classical Boolean computation can be
implemented on a quantum circuit using reversible gate primitives.
The key building block is the **Toffoli gate** (CCX) - a three-qubit
controlled-controlled-NOT that flips the target qubit if and only if
both control qubits are $|1\rangle$.
Because it is its own inverse ($\text{CCX}^2 = I$) it is reversible,
satisfying the unitarity requirement of quantum gates.

This notebook shows AND and OR built from the Toffoli, then implements
Feynman's reversible full adder (1985) which computes the sum and carry
of three input bits using only CCX and CX gates.

---
A single `AerSimulator` backend is shared across all cells.

In [ ]:
"""boolean_gates.ipynb"""

# Cell 01 - Imports and shared AerSimulator backend

from IPython.display import display
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_distribution
from qiskit_aer import AerSimulator

backend = AerSimulator()

print(f"Backend name    : {backend.name}")
print(f"Aer version     : {backend.configuration().backend_version}")
print(f"Max qubits      : {backend.configuration().n_qubits}")
print(f"Max shots       : {backend.configuration().max_shots:,}")
print(f"Simulation type : {backend.configuration().simulator}")
methods = backend.available_methods()
if methods:
    print("Available methods:")
    for method in methods:
        print(f"  {method}")

---
**Cell 02 - AND gate via the Toffoli (CCX).**
The Toffoli gate directly implements AND: it flips an ancilla qubit
(q2, initialized to $|0\rangle$) if and only if both inputs q0 and q1 are $|1\rangle$.
Only q2 is measured; q0 and q1 are left untouched (reversibility).

Change `[0, 1]` to `[1, 0]` in either `initialize` call to set that input to False
and test all four input combinations.

In [ ]:
# Cell 02 - AND gate: CCX(q0, q1, q2) where q2 is ancilla initialized to |0>
# Result on q2: q0 AND q1
# [0,1] = True (|1>),  [1,0] = False (|0>)

qc = QuantumCircuit(3, 1)
qc.initialize([0, 1], 0)  # q0 = True
qc.initialize([0, 1], 1)  # q1 = True
qc.barrier()
qc.ccx(0, 1, 2)  # q2 = q0 AND q1
qc.barrier()
qc.measure(2, 0)

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend)).result()
display(plot_distribution(result.get_counts(qc)))

---
**Cell 03 - OR gate via De Morgan's law.**
There is no direct quantum OR gate, but De Morgan's law gives a construction
using only NOT and AND:

$$a \lor b = \lnot(\lnot a \land \lnot b)$$

The circuit implements this in four stages:
1. X on q0 and q1: compute $\lnot a$ and $\lnot b$
2. X on q2 (ancilla): pre-set ancilla to $|1\rangle$
3. CCX(q0, q1, q2): if $\lnot a$ AND $\lnot b$, flip q2 back to $|0\rangle$; otherwise q2 stays $|1\rangle$
4. X on q0 and q1: restore the original inputs

The result on q2 is $\lnot(\lnot a \land \lnot b) = a \lor b$.

Change `[1, 0]` to `[0, 1]` in either `initialize` call to set that input to True.

In [ ]:
# Cell 03 - OR gate via De Morgan's law: a OR b = NOT(NOT(a) AND NOT(b))
# X on inputs and pre-set ancilla, then Toffoli, then restore inputs
# [1,0] = False (|0>),  [0,1] = True (|1>)

qc = QuantumCircuit(3, 1)
qc.initialize([1, 0], 0)  # q0 = False
qc.initialize([1, 0], 1)  # q1 = False
qc.barrier()
qc.x(0)  # NOT q0
qc.x(1)  # NOT q1
qc.x(2)  # ancilla -> |1>
qc.barrier()
qc.ccx(0, 1, 2)  # if NOT(q0) AND NOT(q1): flip ancilla to |0>
qc.barrier()
qc.x(0)  # restore q0
qc.x(1)  # restore q1
qc.barrier()
qc.measure(2, 0)

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend)).result()
display(plot_distribution(result.get_counts(qc)))

---
**Cells 04 - Feynman's reversible full adder (1985).**
A full adder takes three input bits $a$, $b$, $c_{in}$ (carry-in) and produces
a 1-bit sum $s$ and a 1-bit carry-out $c_{out}$ such that
$a + b + c_{in} = 2\,c_{out} + s$.

Feynman's circuit uses four qubits: q0=$a$, q1=$b$, q2=$c_{in}$, q3=ancilla ($|0\rangle$).
The four gates compute:

| Step | Gate | Effect |
|---|---|---|
| 1 | CCX(0,1,3) | q3 $\leftarrow$ $a \land b$ (partial carry) |
| 2 | CX(0,1) | q1 $\leftarrow$ $a \oplus b$ |
| 3 | CCX(1,2,3) | q3 $\leftarrow$ q3 $\oplus$ ($(a\oplus b) \land c_{in}$) = $c_{out}$ |
| 4 | CX(1,2) | q2 $\leftarrow$ $(a\oplus b)\oplus c_{in}$ = $s$ |

After the circuit: q2 holds the **sum** bit and q3 holds the **carry-out**.
The circuit is fully reversible - running it twice restores all inputs.

In [ ]:
# Cell 04 - Feynman's Full Adder circuit diagram
# q0=a, q1=b, q2=c_in, q3=ancilla
# After circuit: q2=sum, q3=carry_out

qc = QuantumCircuit(4)
qc.ccx(0, 1, 3)
qc.cx(0, 1)
qc.ccx(1, 2, 3)
qc.cx(1, 2)
qc.measure_all()

display(qc.draw(output="mpl"))

---
**Cell 05 - Full adder truth table.**
All eight input combinations are simulated.
In Qiskit's measurement string the qubits are ordered right-to-left
(q3 q2 q1 q0), so `result[0]` is q3 (carry-out) and `result[1]` is q2 (sum).

In [ ]:
# Cell 05 - Truth table for Feynman's Full Adder
# Qiskit result string order: q3 q2 q1 q0  (right-to-left)
# result[0] = q3 = carry_out,  result[1] = q2 = sum


def full_adder(a, b, c_in):
    """Simulate Feynman's full adder for inputs a, b, c_in.
    Returns the Qiskit measurement result string.
    """
    qc = QuantumCircuit(4)
    if a == 1:
        qc.initialize([0, 1], 0)
    if b == 1:
        qc.initialize([0, 1], 1)
    if c_in == 1:
        qc.initialize([0, 1], 2)
    qc.ccx(0, 1, 3)
    qc.cx(0, 1)
    qc.ccx(1, 2, 3)
    qc.cx(1, 2)
    qc.measure_all()
    result = backend.run(transpile(qc, backend)).result()
    return list(result.get_counts(qc))[0]


print("| c_in | b | a |  | c_out | s |")
print("|------|---|---|  |-------|---|")
for c_in in [0, 1]:
    for b in [0, 1]:
        for a in [0, 1]:
            bits = full_adder(a, b, c_in)
            c_out = bits[0]  # q3 = carry_out
            s = bits[1]  # q2 = sum
            print(f"|  {c_in}   | {b} | {a} |  |   {c_out}   | {s} |")